# Sales · 03 Gold
Owned by: Sales lead. Reads from other domains only through shortcuts.

In [ ]:
# Domain: sales · owner: Sales lead
# Attach this notebook to the lakehouse "lh_sales" in the workspace "Solvi Sales".
DOMAIN = "sales"
LANDING_PATH = f"Files/landing/{DOMAIN}"   # upload this domain's CSV files here
TABLE_FORMAT = "delta"
LOCAL_PREFIX = ""                            # local testing sets this to f"{DOMAIN}_"
AS_OF = "2026-10-31"                         # demo data ends here; use the current date with live data

from pyspark.sql import functions as F, Window as W

def write_table(df, name):
    (df.write.format(TABLE_FORMAT).mode("overwrite").option("overwriteSchema", "true").saveAsTable(LOCAL_PREFIX + name))
    print(f"wrote {name:<30} {df.count():>9,} rows")

def read_table(name):                         # a table this domain owns
    return spark.read.table(LOCAL_PREFIX + name)

def read_shared(domain, name):                # another domain's table, through a OneLake shortcut named <domain>_<name>
    return spark.read.table(f"{domain}_{name}")

## Proposal outcome per proposed product
Every proposed line gets exactly one outcome, valued at the signed price.

In [ ]:
q, ql = read_table("silver_proposal"), read_table("silver_proposal_line")
so, sol = read_table("silver_sales_order"), read_table("silver_sales_order_line")
write_table(read_table("silver_seller"), "dim_seller")
ol = (sol.filter(F.col("proposal_line_no").isNotNull())
        .select("proposal_id", F.col("proposal_line_no").alias("line_no"), F.col("product_code").alias("ordered_product"),
                F.col("qty").alias("ordered_qty"), F.col("line_total_nok").alias("ordered_value_nok")))
fql = (ql.join(q.select("proposal_id","customer_id","seller_id","created_date","status","loss_reason"), "proposal_id")
         .join(so.select("proposal_id","order_status"), "proposal_id", "left").join(ol, ["proposal_id","line_no"], "left"))
outcome = (F.when(F.col("status") == "Open", "Open").when(F.col("status") != "Won", "Proposal not accepted")
  .when(F.col("order_status") == "Cancelled", "Contract cancelled").when(F.col("ordered_product").isNull(), "Component dropped")
  .when(F.col("ordered_product") != F.col("product_code"), "Component swapped").when(F.col("ordered_qty") < F.col("qty"), "System downsized")
  .when(F.col("ordered_qty") > F.col("qty"), "System expanded").otherwise("Installed as proposed"))
fact_proposal_line = (fql.withColumn("outcome", outcome)
  .withColumn("ordered_value_nok", F.coalesce("ordered_value_nok", F.lit(0)).cast("decimal(14,2)"))
  .withColumn("changed_value_nok", F.when(F.col("outcome").isin("Proposal not accepted","Component dropped","System downsized","Contract cancelled"),
              F.greatest(F.lit(0), F.col("line_total_nok") - F.col("ordered_value_nok"))).otherwise(F.lit(0)).cast("decimal(14,2)")))
write_table(fact_proposal_line, "fact_proposal_line")

## Proposals and follow-up

In [ ]:
acts = read_table("silver_activity")
first_fu = acts.filter("status = 'Completed'").groupBy("proposal_id").agg(F.min("activity_date").alias("first_followup_date"))
fact_proposal = (q.join(first_fu, "proposal_id", "left")
  .withColumn("days_to_first_followup", F.datediff("first_followup_date", "sent_date"))
  .withColumn("age_days", F.when(F.col("status") == "Open", F.datediff(F.lit(AS_OF), "sent_date")))
  .withColumn("is_signed", (F.col("status") == "Won").cast("int")))
write_table(fact_proposal, "fact_proposal")
write_table(sol.join(so.select("order_id","customer_id","order_date","order_status"), "order_id"), "fact_order_line")
write_table(read_table("silver_invoice").join(so.select("order_id","customer_id"), "order_id"), "fact_invoice")

## Sales history across both ERP systems
The old ERP is frozen. Its lines are mapped to current numbers through crosswalk tables; lines that cannot be mapped keep their old numbers and are flagged, never dropped.

In [ ]:
leg = (read_table("silver_legacy_order_line").join(read_table("silver_legacy_product"), "old_item_no", "left")
  .join(read_table("silver_product_xref"), "old_item_no", "left").join(read_table("silver_customer_xref"), "old_customer_no", "left")
  .select(F.lit("Legacy ERP").alias("source"), F.col("legacy_order_no").alias("order_no"), "order_date", "customer_id", "old_customer_no",
          "product_code", "old_item_no", "description", "category", "qty", "amount_nok"))
cur = (sol.join(so.filter("order_status <> 'Cancelled'").select("order_id","customer_id","order_date"), "order_id")
  .join(read_shared("core", "dim_product").select("product_code", F.col("product_name").alias("description"), "category"), "product_code")
  .select(F.lit("Current ERP").alias("source"), F.col("order_id").alias("order_no"), "order_date", "customer_id",
          F.lit(None).cast("string").alias("old_customer_no"), "product_code", F.lit(None).cast("string").alias("old_item_no"),
          "description", "category", "qty", F.col("line_total_nok").alias("amount_nok")))
fact_sales_history = leg.unionByName(cur).withColumn("customer_mapped", F.col("customer_id").isNotNull())
write_table(fact_sales_history, "fact_sales_history")